# Lesson 31 · 🎨 Distributions visualized — the shapes randomness leaves behind: PDF, CDF and the cross-entropy loss

> **🎨 + module gate**: this lesson wraps up L27–L30 in pictures; it's mostly about reading plots. Once you're done, you're ready for Audio DSP (L32).

Together with `s1–s4`, six plots tie together frequency convergence, distribution shapes, standardization, softmax and the cross-entropy loss. The intuition behind the classification head's `softmax_cross_entropy` comes from the last two plots.

**Goal**: draw six plots with `matplotlib` — the law-of-large-numbers convergence curve, uniform vs normal PDFs, the Gaussian bell with its standard-deviation bands, before/after z-scores, softmax bars, and the cross-entropy loss curve — plus hand-written visualizations of Shannon entropy and the CDF.



## Learning goals
1. Use the sample mean and the law of large numbers to understand "why frequencies converge"
2. Tell the PDF, CDF and PMF apart, and what softmax and cross-entropy each do
3. Write `shannon_entropy(p)` by hand and understand how it relates to KL divergence and cross-entropy
4. Know where the temperature parameter, z-scores and numerical integration show up in these distribution plots
**Why it matters for Aurora**: the mathematical intuition behind the classification head's `softmax_cross_entropy` comes from the last two plots; the Shannon-entropy curves explain why minimizing cross-entropy is equivalent to minimizing KL divergence; the CDF plot gives the probabilistic reading of feature normalization.

← **Previous lesson** · [L30 · Softmax and cross-entropy](L30_softmax_crossentropy.ipynb)

> Last lesson covered **softmax and cross-entropy**: the output layer and loss of a classifier, with the gradient derived by hand.
> This lesson explores **distributions visualized**.

## This lesson's story: let randomness leave a shape

A single random event is noise; repeat it many times and frequencies converge to probabilities, and the samples pile up into a recognizable distribution. This lesson draws every link of that chain: the convergence curve of the law of large numbers (LLN), the standard-deviation (SD) bands of the normal distribution, before-and-after z-score standardization, and softmax squashing raw scores into probabilities that cross-entropy then turns into a loss.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
from sviz import (style, law_of_large_numbers, dist_compare,
                  gaussian, zscore_demo, softmax_bars, cross_entropy_curve)
style(); print('sviz ready')

## Into the lab: how probabilities emerge from samples

Watch two control variables: `n` (the number of samples) decides how stable the frequency estimate is, and `sigma` (the standard deviation) decides how wide the distribution is. Turn up `n` and the frequency curve settles faster; turn up `sigma` and the bell curve spreads out and its peak drops.

## Hands-on: run random code more than once

Run once with a fixed seed and note the frequency estimate at `n=100`; then change `n` to `10000` and compare how far each result is from the true probability 1/6.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    print(f'n={n:5d}  frequency estimate={estimate:.4f}  (theory 1/6≈0.1667)')


## Code experiment: repeat the same random experiment many times

This experiment shows how sample size tames the wobble in a frequency: every time `n` grows 10×, the variance of the estimate across repeated runs shrinks by about 10×.

In [ ]:
import numpy as np

# Law of large numbers: repeat the experiment 5 times and watch the wobble in the frequency estimate narrow as n grows
for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        rolls = rng.integers(1, 7, size=n)
        estimates.append(np.mean(rolls == 6))
    spread = max(estimates) - min(estimates)
    print(f'n={n:4d}  5 estimates={np.round(estimates, 3)}  range={spread:.4f}')


## 1. The law of large numbers: frequency converges to probability (see s1)

Read from left to right, the six plots form one chain of understanding: frequency convergence (plot 1) shows that probabilities can be estimated from repeated experiments; distribution shapes (plots 2–3) describe the statistical structure of data; z-score standardization (plot 4) puts features on different scales into the same units; softmax (plot 5) maps raw scores to valid probabilities; and cross-entropy (plot 6) turns the gap between predicted probabilities and the true label into a differentiable loss. This chain is the mathematical backbone that takes an ML model from raw data to a training signal — each plot is one node of the training pipeline, and drop any step and everything after it stops making sense.

## 1a. The LLN convergence curve by hand (`np.cumsum`)

The core formula of the law-of-large-numbers convergence curve takes just one line: `running = np.cumsum(flips) / np.arange(1, n+1)`.
`np.cumsum` takes the running (prefix) sum of the flip results (0/1); divide by the number of flips so far and you get the running mean.
sviz uses exactly the same formula inside — below we write it out explicitly first, then draw the full plot with sviz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 2000
flips = rng.integers(0, 2, size=n)  # fair coin: 0=tails, 1=heads

# the LLN running mean in one line
running = np.cumsum(flips) / np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(running, color='steelblue', lw=0.8, label='running mean (frequency)')
ax.axhline(0.5, color='tomato', lw=1.5, ls='--', label='true probability p=0.5')
ax.set_xlabel('number of flips n'); ax.set_ylabel('frequency'); ax.legend()
ax.set_title('Law of large numbers: straight from np.cumsum')
plt.tight_layout(); plt.show()

# check: by n=2000 the deviation should be < 5%
assert abs(running[-1] - 0.5) < 0.05, f'frequency deviation too large: {running[-1]:.4f}'

### A detail: why `np.arange(1, n+1)` and not `np.arange(n)`?

Plenty of beginners look at `running = np.cumsum(flips) / np.arange(1, n + 1)` and wonder: why not `np.arange(n)`?

**The numerical difference**:
- `np.arange(n)` produces `[0, 1, 2, ..., n-1]` — the first element is **0**
- `np.arange(1, n+1)` produces `[1, 2, 3, ..., n]` — all positive integers

**What actually happens**:
```python
flips = np.array([1, 0, 1])
cumsum = np.cumsum(flips)  # [1, 1, 2]

# wrong
cumsum / np.arange(3)      # [1, 1, 2] / [0, 1, 2] → [inf, 1.0, 1.0] ❌ the first value blows up!

# right
cumsum / np.arange(1, 4)   # [1, 1, 2] / [1, 2, 3] → [1.0, 0.5, 0.67] ✓
```

**The logic**: the running mean is "the average after the i-th flip", i.e. the sum of the first i numbers divided by i.
- 1st flip: [1] / 1 = 1.0
- 2nd flip: [1+0] / 2 = 0.5
- 3rd flip: [1+0+1] / 3 ≈ 0.67

Because i starts at 1, the denominator has to be `[1, 2, 3, ...]`. `np.arange(n)` starts at 0, so the very first term triggers a "divide by zero".

In [ ]:
law_of_large_numbers();

## 2. Uniform distribution vs normal distribution (see s3)

In [ ]:
dist_compare();

## 3. The Gaussian bell + standard-deviation bands (see s3)

### The geometric meaning of the standard deviation σ: the bell's "width parameter"

The standard deviation measures how "spread out" the data is. For a normal distribution, σ directly controls how wide or narrow the bell curve is.

**From the formula**:
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

When σ increases:
- $(x-\mu)^2/\sigma^2$ in the exponent gets smaller (slower decay)
- the curve gets "flatter" and "wider"
- the factor in front, $1/(\sigma\sqrt{2\pi})$, gets smaller (lower height)

When σ decreases:
- $(x-\mu)^2/\sigma^2$ in the exponent gets larger (faster decay)
- the curve gets "sharper" and "narrower"
- the factor in front gets larger (greater height)

**The geometric meaning of the 68-95-99.7 rule**:
- the area between ±1σ = 68% (the "hillside" in the middle)
- the area between ±2σ = 95% (where most of the data sits)
- the area between ±3σ = 99.7% (almost all of the data)

These three numbers come from integration; they are fixed by the normal distribution itself, not chosen at random.

**What the z-score does**:
$$z = \frac{x - \mu}{\sigma}$$

What does this transformation do?
- subtract μ first: move the curve's center to 0
- then divide by σ: rescale the "width" to a standard ruler

After the transformation every normal distribution (whatever its original μ and σ) becomes the "standard normal" (μ=0, σ=1). That way any normal distribution can be read with the same 68-95-99.7 yardstick.

In [ ]:
gaussian(0, 1);

## 4. Before and after z-score standardization (see s2)

**🔗 Aurora**: features must be standardized before they go into a model. `normalize_features` in Aurora's training loop subtracts the mean and divides by the standard deviation for every dimension; skip this step and the differences in scale between frequency bands make the gradient too large in high-energy dimensions and close to zero in low-energy ones, so no single learning rate suits every dimension and training becomes unstable. The implementation lives in `src/aurora/audio/transforms.py` (`aurora.audio.normalize_features`).

#### Why does standardization "line up the gradients"? A concrete case of gradient imbalance

In Aurora's audio models, the input features come from different frequency bands with wildly different scales. For example, the energy in a low band might be 1000 and in a high band 0.01, and that directly affects the size of the network's gradients.

**How the imbalance arises**:
the gradient that reaches a weight is proportional to its input. Suppose some layer of the model computes `y = w·x + b`; then:
$$\frac{\partial L}{\partial w} = x \cdot \frac{\partial L}{\partial y}$$

If `x` is large, the weight's gradient is large; if `x` is tiny, the gradient is tiny too. This leads to:
- high-energy dimensions (e.g. low frequencies): huge gradients, the weights w oscillate wildly
- low-energy dimensions (e.g. high frequencies): tiny gradients, the weights w update slowly

Training becomes unstable, and no single learning rate fits every dimension.

**The fix — standardization**:
$$z = \frac{x - \mu}{\sigma}$$
moves every feature onto a "standard scale" with mean 0 and standard deviation 1. Now:
- almost all feature values lie within −3 to 3 (the 68-95-99.7 rule)
- gradient sizes are put on the same scale
- a single learning rate works for every dimension

```python
# compare
x = np.array([1000.0, 0.01])   # values differ by a factor of 10⁵
z = (x - x.mean()) / x.std()   # [1., -1.] — now the values are comparable
```

The comparison plot below shows this change at a glance.

In [ ]:
zscore_demo();

## 5. softmax: scores → probabilities (see s4)

**🔗 Aurora**: the classification head's `softmax_cross_entropy` first passes the last layer's logit vector through softmax to get a valid probability distribution (every entry non-negative, summing to 1), then takes the negative log of the true class's probability to get the loss. The two steps are combined using the log-sum-exp trick, which keeps `np.exp(logit)` from overflowing on large values (numerical overflow). The implementation lives in `src/aurora/llm/sample.py` (`aurora.llm.softmax_cross_entropy`).

In [ ]:
softmax_bars((2.0, 1.0, 0.1));

### Key trick: softmax numerical stability — why subtract the maximum?

Computing softmax straight from its definition runs into a numerical-overflow disaster:
$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**The problem**: when z contains large numbers (say 1000), `np.exp(1000)` far exceeds what a float can hold and returns `inf` (float64 overflows once the input passes about 709).

**The fix**: a mathematical trick — subtract the maximum `c = max(z)`:
$$\text{softmax}(z_i) = \frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$$

**Why doesn't this change the result?** Because of a property of exponentials:
$$e^{z_i - c} = \frac{e^{z_i}}{e^c}$$

So numerator and denominator are both divided by $e^c$, and the ratio is unchanged — it's an equivalent transformation.

**Example**:
```
z = [1000, 1001, 1002]
naively: softmax with exp(1000), exp(1001), exp(1002) → inf (overflow)
subtract the max: z' = [1000-1002, 1001-1002, 1002-1002] = [-2, -1, 0]
         softmax with exp(-2), exp(-1), exp(0) ≈ [0.0900, 0.2447, 0.6652] (safe)
```

**General rule**: in any numerical computation involving `exp`, subtract the largest input first to avoid overflow. This is the **log-sum-exp trick**, and it shows up in softmax, cross-entropy, LSTMs and more.

## 6. Cross-entropy: the more confident and correct, the lower the loss (see s4)

In [ ]:
cross_entropy_curve();

## 7. Visualizing Shannon entropy — the shape of uncertainty

Shannon entropy H(p) = −Σ pᵢ log₂ pᵢ measures "how uncertain" a probability distribution is.
The two plots show:
- **Left**: H(p) of a two-outcome distribution as p varies — it peaks at p=0.5 (1 bit) and falls to 0 as p→0 or p→1
- **Right**: entropy H(p) compared with cross-entropy H(p,q) — H(p,q) = H(p) + KL(p‖q) ≥ H(p), so minimizing cross-entropy is equivalent to minimizing KL divergence

### Why Shannon entropy? From everyday intuition to the formula

Imagine you're playing roulette at a casino, and the wheel has a few possible setups:

| Setup | Probability distribution | How it feels |
|------|---------|--------|
| probabilities=[1, 0, 0, ...] | outcome already decided | no suspense at all (uncertainty = 0) |
| probabilities=[0.5, 0.5] | coin: 50% heads, 50% tails | some suspense |
| probabilities=[0.25, 0.25, 0.25, 0.25] | four-sided die: each face 1/4 | more suspense |

**Putting a number on uncertainty**: if an event is "surprising", it means it had a small probability. We measure the amount of surprise with −log₂(p):
- p=1 (certain): −log₂(1) = 0 (no suspense)
- p=0.5 (50% chance): −log₂(0.5) = 1 bit (1 bit of information)
- p=0.25 (25% chance): −log₂(0.25) = 2 bits (more surprising)

**Shannon entropy = average surprise**:
$$H(p) = -\sum_{i} p_i \log_2 p_i = \mathbb{E}[-\log_2(p_i)]$$

Not the surprise of one particular trial, but the **average surprise over many trials**.

- Certain distribution [1, 0, ...]: never any suspense, average H = 0
- Uniform two-outcome distribution [0.5, 0.5]: average H = 1 bit
- Uniform N-outcome distribution [1/N, ..., 1/N]: average H = log₂(N) bits

**Why log₂ and not ln?**
Because information theory usually measures in "bits". 1 bit = the information carried by one binary choice (yes/no, 0/1). The base-2 log matches a binary choice, so log₂ is the natural pick. With ln, the unit becomes the "nat" (natural unit of information).

---

### Handling the limit 0 log 0 = 0

When computing H(p) = −Σ pᵢ log₂ pᵢ, if some component pᵢ = 0 (the event never happens), the term pᵢ log₂ pᵢ becomes the indeterminate form "0 × (−∞)". Mathematically we define this term to be 0, because of the limit:

$$\lim_{p \to 0^+} p \log_2 p = 0$$

**Why is this limit 0?** By L'Hôpital's rule:
$$\lim_{p \to 0^+} p \log_2 p = \lim_{p \to 0^+} \frac{\log_2 p}{1/p} = \lim_{p \to 0^+} \frac{1/(p \ln 2)}{-1/p^2} = \lim_{p \to 0^+} \frac{-p}{\ln 2} = 0$$

Intuitively: log₂(p) → −∞, but the p in front goes to 0 faster, so the product still ends up at 0.

An event with probability 0 never happens, so it contributes 0 to the entropy — perfectly consistent.

## Exercise: write `shannon_entropy(p)` by hand

Using the formula **H(p) = −Σ pᵢ log₂ pᵢ**, implement the function below.

Hints:
- Use `np.where` or a boolean mask to skip the pᵢ = 0 terms (0 log 0 is defined as 0).
- The return value is in **bits** (base-2 logarithm).
- When you're done, all the assertions below should pass.

In [ ]:
import numpy as np

def shannon_entropy(p: np.ndarray) -> float:
    """Compute the Shannon entropy of a discrete probability distribution p (in bits).

    Parameters
    ----------
    p : array-like, non-negative entries summing to 1

    Returns
    -------
    float  entropy ≥ 0; largest for a uniform distribution, 0 for a certain one
    """
    # ── write your implementation here ──────────────────────────────────
    raise NotImplementedError('Implement shannon_entropy')
    # ────────────────────────────────────────────────────────


# ── checks ──────────────────────────────────────────────────────────────
try:
    # certain distribution: entropy should be 0
    h0 = shannon_entropy(np.array([1.0, 0.0, 0.0]))
    assert abs(h0) < 1e-9, f'a certain distribution should have entropy 0, got {h0:.6f}'

    # uniform two-outcome distribution: entropy should be 1 bit
    h1 = shannon_entropy(np.array([0.5, 0.5]))
    assert abs(h1 - 1.0) < 1e-9, f'a uniform two-outcome distribution should have entropy 1.0 bit, got {h1:.6f}'

    # uniform four-outcome distribution: entropy should be 2 bits
    h2 = shannon_entropy(np.array([0.25, 0.25, 0.25, 0.25]))
    assert abs(h2 - 2.0) < 1e-9, f'a uniform four-outcome distribution should have entropy 2.0 bits, got {h2:.6f}'

    print('✓ All assertions passed! Your shannon_entropy is correct.')
except (NotImplementedError, TypeError):
    print('⚠ Not implemented yet — fill in the function body and run again.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# ── Left: H(p) curve of a two-outcome distribution ─────────────────────────
p = np.linspace(1e-9, 1 - 1e-9, 500)
H = -p * np.log2(p) - (1 - p) * np.log2(1 - p)

ax1.plot(p, H, color='#2A9D8F', lw=2.5)
ax1.axvline(0.5, color='#E76F51', lw=1.2, ls='--', alpha=0.8)
ax1.scatter([0.5], [1.0], s=60, color='#E76F51', zorder=5)
ax1.annotate('H = 1 bit\n(uniform, most uncertain)', (0.5, 1.0),
             xytext=(0.58, 0.82), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='#E76F51'))
ax1.set_xlabel('probability p')
ax1.set_ylabel('entropy H(p)  [bits]')
ax1.set_title('Shannon entropy of a two-outcome distribution\nH(p) = −p log₂p − (1−p) log₂(1−p)')
ax1.set_ylim(-0.05, 1.2)

# ── Right: entropy vs cross-entropy bar chart ──────────────────────────────
def h_bits(p):
    p = np.asarray(p, float)
    return float(-np.sum(p[p > 0] * np.log2(p[p > 0])))

def hpq_bits(p, q):
    p = np.asarray(p, float)
    q = np.clip(np.asarray(q, float), 1e-15, None)
    return float(-np.sum(p * np.log2(q)))

p_true = np.array([0.7, 0.2, 0.1])
q_good = np.array([0.65, 0.25, 0.10])
q_bad  = np.array([0.10, 0.10, 0.80])

labels = ['H(p)\nentropy (lower bound)', 'H(p, q_good)\nsmall KL', 'H(p, q_bad)\nlarge KL']
vals   = [h_bits(p_true), hpq_bits(p_true, q_good), hpq_bits(p_true, q_bad)]
colors = ['#2A9D8F', '#8AB17D', '#E76F51']

bars = ax2.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.04,
             f'{v:.3f}', ha='center', fontsize=10)
ax2.axhline(vals[0], color='#2A9D8F', ls='--', lw=1.2, alpha=0.6, label='H(p) lower bound')
ax2.set_ylabel('information [bits]')
ax2.set_title('H(p, q) = H(p) + KL(p‖q) ≥ H(p)\nminimizing cross-entropy ≡ minimizing KL divergence')
ax2.legend(fontsize=9)
ax2.set_ylim(0, max(vals) * 1.3)

plt.tight_layout()
plt.show()

print(f'H(p)={vals[0]:.4f}  H(p,q_good)={vals[1]:.4f}  H(p,q_bad)={vals[2]:.4f}')
assert vals[1] >= vals[0] - 1e-9 and vals[2] >= vals[0] - 1e-9
print('✅ H(p,q) ≥ H(p) verified')

---

### Going further: the definition and geometric meaning of KL divergence

KL(p‖q) appears in the cross-entropy formula H(p,q) = H(p) + KL(p‖q). What is it?

**Definition of KL divergence**:
$$\text{KL}(p \| q) = \sum_{i} p_i \log_2 \frac{p_i}{q_i} = \sum_{i} p_i (\log_2 p_i - \log_2 q_i)$$

(written $\sum p_i \ln(p_i / q_i)$ when using the natural log)

**Geometric meaning**: KL divergence measures how "different" two probability distributions are.
- KL(p‖q) = 0 if and only if p = q (identical)
- KL(p‖q) > 0 when p ≠ q (some degree of difference)
- the larger KL(p‖q), the more the distributions differ

**Why does this formula measure a "distance"?**
KL divergence comes from information theory: if the true distribution is p but we encode with the wrong distribution q, the average number of extra bits wasted is exactly KL(p‖q).

**Relationship to cross-entropy**:
$$H(p, q) = -\sum_i p_i \log_2 q_i = -\sum_i p_i \log_2 p_i - \sum_i p_i (\log_2 q_i - \log_2 p_i)$$
$$= H(p) + \text{KL}(p \| q)$$

So minimizing cross-entropy is equivalent to minimizing KL divergence (because H(p) is fixed — it's the entropy of the true labels and can't be changed).

**Why is KL not symmetric?**
- KL(p‖q): the true distribution is p and the model's is q → it weights the events that are likely under p
- KL(q‖p): the other way round → it weights the events that are likely under q

In classification we use KL(p‖q) (p is the true label, q the model's prediction), so that when the model gets the true class's probability wrong, the loss is large.

## 8. Visualizing the cumulative distribution function (CDF) — Φ(x) = P(X ≤ x)

The CDF is the integral of the PDF from −∞ to x, and it fully describes "the probability that the random variable doesn't exceed a given value".
Here we compute the standard normal CDF Φ(x) by numerical integration (the trapezoidal rule) — the same principle as `normal_cdf` in L29, plus a visualization.

> **The 68–95–99.7 rule of thumb**: Φ(1)−Φ(−1) ≈ 68%, Φ(2)−Φ(−2) ≈ 95%, Φ(3)−Φ(−3) ≈ 99.7%

### How trapezoidal integration works, and how accurate it is

To compute the CDF Φ(x) = ∫_{−∞}^{x} f(t) dt we can't use a closed-form formula; we have to integrate numerically. Here we use the **trapezoidal rule**.

**How the trapezoidal rule works**:
split the curve into many small intervals and approximate each interval with a trapezoid:
$$\int_a^b f(t) dt \approx \sum_{i=0}^{n-1} \frac{f(x_i) + f(x_{i+1})}{2} \cdot \Delta x$$

where $\Delta x$ is the spacing between neighboring sample points.

**Example**:
```
the integral of f(t) over [0, 2]:
sample points: x = [0, 1, 2], f(x) = [1, 2, 5], Δx = 1
area ≈ (1+2)/2 × 1 + (2+5)/2 × 1 = 1.5 + 3.5 = 5.0
```

---

### How the code's parameters were chosen

The parameters used in `normal_cdf`:
```python
def normal_cdf(x_eval, mu=0.0, sigma=1.0, n=4000):
    xs  = np.linspace(mu - 6 * sigma, x_eval, n)
    return float(np.trapezoid(gaussian_pdf(xs, mu, sigma), xs))
```

**Why start at `mu - 6*sigma` rather than −∞?**
- In theory the PDF is integrated from −∞ to +∞
- But beyond ±6σ the normal distribution is already tiny (the probability left out beyond −6σ is about 10⁻⁹)
- So leaving that part out costs very little accuracy

**The exact values of the 68-95-99.7 rule**:
$$P(|X - \mu| \leq \sigma) \approx 0.6827$$
$$P(|X - \mu| \leq 2\sigma) \approx 0.9545$$
$$P(|X - \mu| \leq 3\sigma) \approx 0.9973$$

The range −6σ to 6σ already covers more than 99.99% of the probability, so the remaining error is under 0.01%.

**Why use n=4000 sample points?**
- The trapezoidal rule's error falls with the square of n: error ∝ 1/n²
- With 4000 points over [−6σ, 6σ], the spacing is Δx ≈ 0.003, enough to reach 10⁻⁶ accuracy
- That's more than precise enough to check the 68-95-99.7 rule

**How do you know the accuracy is good enough?**
A simple check: see whether the result stays put when you change n (say n=2000 vs n=4000). We'll demonstrate below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gaussian_pdf(x, mu=0.0, sigma=1.0):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

def normal_cdf(x_eval, mu=0.0, sigma=1.0, n=4000):
    xs  = np.linspace(mu - 6 * sigma, x_eval, n)
    return float(np.trapezoid(gaussian_pdf(xs, mu, sigma), xs))

x   = np.linspace(-4, 4, 600)
pdf = gaussian_pdf(x)
cdf = np.array([normal_cdf(xi) for xi in x])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# ── Left: PDF with shaded areas ───────────────────────────────────────────
fill_spec = [(1, '#2A9D8F', 0.55, '±1σ ≈ 68%'),
             (2, '#8AB17D', 0.35, '±2σ ≈ 95%'),
             (3, '#E9C46A', 0.20, '±3σ ≈ 99.7%')]
for sig, color, alpha, label in reversed(fill_spec):
    mask = (x >= -sig) & (x <= sig)
    ax1.fill_between(x[mask], pdf[mask], alpha=alpha, color=color, label=label)
ax1.plot(x, pdf, color=plt.rcParams["text.color"], lw=2)
ax1.set_xlabel('x'); ax1.set_ylabel('PDF f(x)')
ax1.set_title('Standard normal PDF\nthe 68–95–99.7 rule of thumb')
ax1.legend(fontsize=9, loc='upper right')

# ── Right: CDF Φ(x) ──────────────────────────────────────────────────────
ax2.plot(x, cdf, color='#2A9D8F', lw=2.5, label='Φ(x)')
ax2.axhline(0.5, color=plt.rcParams["grid.color"], ls=':', lw=1, alpha=0.6)
sigma_marks = [(1, '#2A9D8F'), (2, '#8AB17D'), (3, '#E9C46A')]
for sig, color in sigma_marks:
    y = normal_cdf(sig)
    ax2.scatter([sig], [y], s=45, color=color, zorder=5)
    ax2.annotate(f'Φ({sig})={y:.4f}', (sig, y),
                 xytext=(sig + 0.1, y - 0.05), fontsize=8.5, color=color)
ax2.set_xlabel('x'); ax2.set_ylabel('CDF Φ(x) = P(X ≤ x)')
ax2.set_title('Standard normal CDF (numerical integration)\nΦ(x) = ∫_{−∞}^{x} f(t) dt')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

p68  = normal_cdf(1)  - normal_cdf(-1)
p95  = normal_cdf(2)  - normal_cdf(-2)
p997 = normal_cdf(3)  - normal_cdf(-3)
print(f'±1σ: {p68:.4f}  ±2σ: {p95:.4f}  ±3σ: {p997:.4f}')
assert abs(p68  - 0.6827) < 0.002
assert abs(p95  - 0.9545) < 0.002
assert abs(p997 - 0.9973) < 0.002
print('✅ CDF numerical integration verified: the 68–95–99.7 rule of thumb')

In [ ]:
import numpy as np

# Accuracy check: use different numbers of sample points and see whether the integral is stable
def gaussian_pdf(x, mu=0.0, sigma=1.0):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

def normal_cdf_with_n(x_eval, n, mu=0.0, sigma=1.0):
    xs = np.linspace(mu - 6 * sigma, x_eval, n)
    return float(np.trapezoid(gaussian_pdf(xs, mu, sigma), xs))

# exact 68-95-99.7 values vs numerical integration
target_sigs = [1, 2, 3]
n_samples = [500, 1000, 2000, 4000]

print("Checking the 68-95-99.7 rule (numerical accuracy test)")
print("=" * 70)
for sig in target_sigs:
    print(f"\nKey value: Φ({sig}) = ∫_{{-∞}}^{sig} f(t) dt")
    print(f"  points     |  integral        |  difference from theory")
    print("  " + "-" * 60)

    for n in n_samples:
        result = normal_cdf_with_n(sig, n)
        # theoretical value (standard normal)
        from scipy.special import erf
        theory = 0.5 * (1 + erf(sig / np.sqrt(2)))
        diff = abs(result - theory)
        print(f"  {n:6d}     |  {result:.8f}  |  {diff:.2e}")

# final check of the rule of thumb
print("\n" + "=" * 70)
print("The 68-95-99.7 rule of thumb (final check)")
print("=" * 70)
p68 = normal_cdf_with_n(1, 4000) - normal_cdf_with_n(-1, 4000)
p95 = normal_cdf_with_n(2, 4000) - normal_cdf_with_n(-2, 4000)
p997 = normal_cdf_with_n(3, 4000) - normal_cdf_with_n(-3, 4000)

print(f"±1σ: {p68:.6f} ≈ 68.27%  ✓")
print(f"±2σ: {p95:.6f} ≈ 95.45%  ✓")
print(f"±3σ: {p997:.6f} ≈ 99.73%  ✓")
print("\nWith 4000 sample points the accuracy reaches 10⁻⁶ — plenty to check the rule of thumb.")

In [ ]:
import numpy as np

def softmax(z):
    e = np.exp(np.asarray(z, float) - np.max(z))
    return e / e.sum()

def cross_entropy(probs, idx):
    return float(-np.log(probs[int(idx)] + 1e-12))

# how the cross-entropy loss changes with confidence
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = np.array([2.0, 1.0, 0.0]) * scale
    p = softmax(z)
    loss = cross_entropy(p, 0)
    print(f'scale={scale:.1f}  P(target class)={p[0]:.4f}  loss={loss:.4f}')


## Parameter experiment: logit gaps and model confidence

In the softmax plot, change the three logits from `[2.0, 1.0, 0.0]` to `[10.0, 5.0, 0.0]` and watch the probability distribution go from fairly spread out (about 0.67 / 0.24 / 0.09) to extremely concentrated (the first close to 1.0). This shows what the "temperature" parameter does: dividing softmax's inputs by a temperature T is the same as scaling all the gaps by the same factor — with T < 1 (wider gaps) the model's output is more confident, with T > 1 it's smoother. It's commonly used in knowledge distillation (KD) and when tuning sampling in text generation. Then change the normal distribution's `sigma` from 1 to 3 and confirm that the share inside ±1σ stays at 68%, while the bell widens and its peak drops.

#### The maths of the temperature parameter, and where it's used

Dividing by a temperature T inside softmax looks like a "magic" tuning knob, but there's a clear mathematical reason behind it.

**Core formula**:
$$\text{softmax}_T(z_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

**Mathematical effect**:
- `T = 1`: standard softmax
- `T < 1` (say 0.1): equivalent to "magnifying the gaps between inputs" → a more concentrated distribution
- `T > 1` (say 5.0): equivalent to "shrinking the gaps between inputs" → a more uniform distribution (closer to random)

**Why?** From a Taylor-expansion point of view, how concentrated softmax's output is depends on the **variance** of its inputs. Dividing by T scales that variance directly:
- smaller variance (T > 1) → smaller differences between entries → a more uniform softmax output (higher entropy)
- larger variance (T < 1) → larger differences between entries → a sharper softmax output (one class close to 1)

**Where it's used in practice**:

1. **Knowledge distillation (KD)**:
   - The large model's (the teacher's) logits are softened with a high temperature (say T=5.0) → a softer distribution that also shows how the wrong classes rank
   - The small model (the student) learns to match this soft distribution at the same temperature → it picks up a "gentler, more forgiving" view
   - At inference time T=1.0 is used to keep predictions accurate

2. **Sampling in text generation**:
   - Sampling at a low temperature (T<1) → more deterministic, more coherent output
   - Sampling at a high temperature (T>1) adds diversity → stops the model always picking the top-scoring word

3. **Model calibration**:
   - If the model's probabilities are too "confident" (often predicting close to 1.0 but actually wrong), adjust with a higher temperature
   - If the model is "not confident enough" (always around 0.6), adjust with a lower temperature

**In code**: the next cell demonstrates the effect of T.

In [ ]:
import numpy as np

def softmax(z):
    e = np.exp(np.asarray(z, float) - np.max(z))
    return e / e.sum()

# the temperature effect: logit ÷ temperature
# T < 1 → gaps magnified → concentrated distribution (confident); T > 1 → gaps squeezed → smooth distribution (random)
logits = np.array([2.0, 1.0, 0.0])
for T in [0.1, 0.5, 1.0, 2.0, 5.0]:
    p = softmax(logits / T)
    print(f'T={T:.1f}  probabilities={np.round(p, 3)}')
print('→ the smaller T, the more concentrated the distribution (confident); the larger T, the more uniform (random).')
print('→ knowledge distillation and sampling in text generation both use the temperature parameter.')

## Wrap-up

You can now draw the frequency-convergence curve with `np.cumsum`, write softmax by hand with `np.exp` and check that its output sums to 1, and compute the cross-entropy loss with `-np.log`. These six plots map onto three nodes of Aurora's training loop: `aurora.audio.normalize_features` (`src/aurora/audio/transforms.py`) matches the standardization plot, `aurora.llm.softmax_cross_entropy` (`src/aurora/llm/sample.py`) matches the last two plots, and the loss curve is used to debug gradient behavior. Next up is L32 (NumPy for signals: building a 16 kHz time axis with np.arange), the start of Module 5, Audio DSP; when the FFT processes audio, the distribution of Fourier-coefficient magnitudes connects directly with the normal-distribution intuition from here.

In [ ]:
# Quick check: same experiment — the larger the sample, the steadier the frequency
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.4f}  (theory 1/6≈{1/6:.4f})')
print('→ by n=3000 the error is already small — this is the foundation of statistical inference.')


In [ ]:
# Standalone maths assertions: checking core properties of probability distributions (no functions to implement)
import numpy as np

# 1. numerical integral of the normal PDF ≈ 1 (a probability density must be normalized)
xs = np.linspace(-6, 6, 10_001)
pdf_vals = np.exp(-0.5 * xs**2) / np.sqrt(2 * np.pi)
area = np.trapezoid(pdf_vals, xs)
assert abs(area - 1.0) < 1e-5, f"the normal PDF's area should be 1, got {area:.8f}"

# 2. Shannon entropy: H([0.5,0.5])=1 bit, a certain distribution has entropy ≈0
p_uniform = np.array([0.5, 0.5])
H_unif = float(-np.sum(p_uniform * np.log2(p_uniform)))
assert np.isclose(H_unif, 1.0, atol=1e-12), f"H([0.5,0.5]) should be 1 bit, got {H_unif}"
# uniform four-outcome distribution: H = log2(4) = 2 bits
p_4 = np.full(4, 0.25)
H_4 = float(-np.sum(p_4 * np.log2(p_4)))
assert np.isclose(H_4, 2.0, atol=1e-12), f"H([0.25x4]) should be 2 bits, got {H_4}"

# 3. softmax output sums to 1; equal logits → uniform probabilities
def _sm(z): e = np.exp(z - z.max()); return e / e.sum()
sm_test = _sm(np.array([2.0, 1.0, 0.1]))
assert np.isclose(sm_test.sum(), 1.0, atol=1e-12)
sm_uniform = _sm(np.array([5.0, 5.0, 5.0]))
assert np.allclose(sm_uniform, [1/3, 1/3, 1/3], atol=1e-12)

# 4. the 68-95-99.7 rule (checked with a large LLN sample)
rng = np.random.default_rng(0)
samples = rng.normal(0, 1, 1_000_000)
assert abs(np.mean(np.abs(samples) <= 1) - 0.6827) < 0.005
assert abs(np.mean(np.abs(samples) <= 2) - 0.9545) < 0.005

print("✅ normal PDF area = 1  ✅ Shannon-entropy properties  ✅ softmax normalization  ✅ 68-95-99.7 rule")
print(f"   H([0.5,0.5])={H_unif:.4f} bit   normal PDF integral={area:.8f}")


## A mental switch before L32: from probability plots to arrays of sound

L27–L31 were about "what shape do lots of samples leave behind": frequencies converge to probabilities, samples pile up into distributions, and logits become class probabilities through softmax.

From L32 on we switch to the Audio DSP module, and the question becomes:

```text
a piece of sound = lots of sample points
lots of sample points + a time axis = a signal you can plot, cut into frames and feed to the FFT
```

In other words, L31's probability plots trained you to see "the overall shape of an array"; L32 will train you to build that array's horizontal axis first.

Before L32, do 3 very quick recalls:

- Look back at L03: a spectrogram's horizontal axis is time, its vertical axis is frequency, and brightness is energy.
- Look back at L01/L02: one second of 16 kHz sound has 16,000 sample points.
- Take one question with you into L32: `at what second does sample point n occur?`

Answer that, and the STFT, mel and MFCC that follow will rest on solid ground.


### Self-check before L32 (just read it)

- [ ] I can name the three axes of an L03 spectrogram: time / frequency / energy.
- [ ] I can say that 16000 Hz means 16,000 sample points per second.
- [ ] I know the core of L32 isn't building a spectrum first, but first building the time axis `t`.

If you're unsure about two or more of these, read the "L31 → L32 switch" section in `docs/current/course/prep-checklist.md` before moving on to the next lesson.


In [ ]:
# ✏️ Self-review for this lesson
l31_review = {
    "lln_curve_seen":          None,  # seen the frequency-convergence curve? True/False
    "softmax_sum_one":         None,  # can show by hand that softmax output sums to 1? True/False
    "cross_entropy_understood": None,  # understand what the cross-entropy loss curve means? True/False
    "shannon_entropy_seen":    None,  # seen the Shannon-entropy curve? True/False
    "l32_prep_read":           None,  # read the L31→L32 switch checklist? True/False
}

unfilled = [k for k, v in l31_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l31_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L31 fully cleared! On to L32: NumPy for signals')


---
⬇️ **Pass check**: you've read the six plots and the wrap-up; tick off the self-review before moving on to L32.


---

→ **Next lesson** · [L32 · NumPy for signals](https://github.com/djzoom/AURORA/blob/main/notebooks/5_audio_dsp/L32_numpy_signals.ipynb) *(Chinese)*

> Next lesson covers **NumPy for signals**: building a 16 kHz time axis with np.arange / linspace and stamping every sample point with its time.